# Day 20 — Hyperparameter tuning and model optimisation

Run each cell with **Shift + Enter**, from the top. We use `student_data_150.csv`: 8 real students and 142 **synthetic** ones.

## 1. Hyperparameter tuning fundamentals

In [ ]:
# Get the student dataset: Colab starts with an empty folder. If the download fails, upload student_data_150.csv
# with the Files panel (the folder icon on the left) and run this cell again
import os
import urllib.request

URL = "https://raw.githubusercontent.com/neerajb1/learnforge-notebooks/main/ai-ml-120/day-020/data/student_data_150.csv"
if not os.path.exists("student_data_150.csv"):
    urllib.request.urlretrieve(URL, "student_data_150.csv")

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# The 23 test students are set aside first and stay untouched all day; tuning happens on the other 127
students = pd.read_csv("student_data_150.csv")
features = ["StudyHours", "Attendance", "PreviousScore"]
pool, test = train_test_split(students, test_size=0.15, random_state=42, stratify=students["Passed"])
X, y = pool[features], pool["Passed"].astype(int)

# Hyperparameter: C, chosen by us before training. Parameters: the coefficients, learned from the data
def build_model(C):
    return Pipeline([("scaler", StandardScaler()), ("classifier", LogisticRegression(C=C, max_iter=5000))])

for C in [0.01, 1, 100]:
    clf = build_model(C).fit(X, y).named_steps["classifier"]
    print(f"C={C:<5} learned coefficients {clf.coef_[0].round(3)}  intercept {clf.intercept_[0]:.3f}")

C=0.01  learned coefficients [0.211 0.127 0.157]  intercept 0.914
C=1     learned coefficients [1.112 0.566 0.71 ]  intercept 1.336
C=100   learned coefficients [1.212 0.607 0.755]  intercept 1.394


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

# Tuning C two ways: on one validation split (as manual tuning often does) and by 5-fold cross-validation
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=0, stratify=y)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("C        one split  5-fold CV (mean ± std)")
for C in [0.001, 0.01, 0.1, 1, 10, 100]:
    one = build_model(C).fit(X_tr, y_tr).score(X_val, y_val)
    s = cross_val_score(build_model(C), X, y, cv=cv)
    print(f"{C:<8} {one:>9.3f}  {s.mean():.3f} ± {s.std():.3f}")

C        one split  5-fold CV (mean ± std)
0.001        0.692  0.709 ± 0.014
0.01         0.692  0.709 ± 0.014
0.1          0.885  0.803 ± 0.042
1            0.846  0.796 ± 0.062


10           0.846  0.788 ± 0.073


100          0.846  0.788 ± 0.073


In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Underfitting and overfitting: a decision tree's max_depth, training accuracy vs cross-validated accuracy
print("max_depth  train  CV     leaves")
for depth in [1, 2, 3, 4, 6, 10, None]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    train_acc = tree.fit(X, y).score(X, y)
    cv_acc = cross_val_score(tree, X, y, cv=cv).mean()
    print(f"{str(depth):<9}  {train_acc:.3f}  {cv_acc:.3f}  {tree.get_n_leaves():>3}")

max_depth  train  CV     leaves
1          0.772  0.709    2
2          0.787  0.740    4
3          0.819  0.717    7
4          0.843  0.725   11
6          0.913  0.725   18
10         0.992  0.710   32


None       1.000  0.717   33


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# A random forest's max_depth, tuned by cross-validation on the training students
print("max_depth  CV accuracy")
for depth in [2, 4, 6, 10, None]:
    forest = RandomForestClassifier(n_estimators=100, max_depth=depth, random_state=42)
    s = cross_val_score(forest, X, y, cv=cv)
    print(f"{str(depth):<9}  {s.mean():.3f} ± {s.std():.3f}")

max_depth  CV accuracy


2          0.749 ± 0.060


4          0.788 ± 0.066


6          0.820 ± 0.070


10         0.804 ± 0.063


None       0.804 ± 0.063


In [ ]:
from sklearn.model_selection import ParameterGrid

# Why manual tuning doesn't scale: count the combinations, then the model fits with 5-fold CV
grid = {"n_estimators": [50, 100, 200, 300, 500], "max_depth": [3, 5, 10, None],
        "min_samples_split": [2, 5, 10], "max_features": ["sqrt", "log2", None]}
combinations = len(ParameterGrid(grid))
print("Combinations:", " x ".join(str(len(v)) for v in grid.values()), "=", combinations)
print("Fits with 5-fold CV:", combinations * 5, "(plus one refit of the best)")

Combinations: 5 x 4 x 3 x 3 = 180
Fits with 5-fold CV: 900 (plus one refit of the best)
